# DA6401 Assignment 1 — Complete W&B Logging Notebook

Run every cell top-to-bottom. All plots and metrics log to W&B automatically.

| Q | Logged |
|---|---|
| 2.1 | Table: 5 images × 10 classes + distribution chart |
| 2.2 | 100-run sweep → parallel coordinates |
| 2.3 | 4 optimizer convergence curves |
| 2.4 | Gradient norms (sigmoid vs relu, 3 vs 5 layers) |
| 2.5 | Dead neuron % + activation histograms |
| 2.6 | MSE vs Cross-Entropy training curves |
| 2.7 | Train vs test accuracy gap (12 configs) |
| 2.8 | Confusion matrix + per-class accuracy + top-10 confused pairs |
| 2.9 | 5-neuron gradient traces: zeros vs xavier |
| 2.10 | 3 Fashion-MNIST transfer runs |

## ⚙️ CELL 0 — CONFIG  ← Edit only this cell

In [3]:
# ============================================================
#  EDIT THESE TWO LINES ONLY
WANDB_PROJECT = "da6401_assignment1"   # your W&B project name
WANDB_ENTITY  = "da25m024-iitm"                 # your W&B username, or None
# ============================================================

import sys, os

for _p in ["../src", "./src", "."]:
    _abs = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_abs, "ann")) and _abs not in sys.path:
        sys.path.insert(0, _abs)
        print(f"Added to sys.path: {_abs}")
        break

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import types, json
import wandb
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)
from ann.neural_network      import NeuralNetwork
from ann.objective_functions import get_loss
from utils.data_loader       import load_data

np.random.seed(42)
print('✅  All imports OK')

Added to sys.path: c:\Users\theri\Desktop\sem 2\DL\assignment_1\src
✅  All imports OK


## 🔧 Helpers — run before any Q cell

In [4]:
def make_args(**kwargs):
    defaults = dict(
        dataset='mnist', epochs=10, batch_size=64,
        learning_rate=0.001, weight_decay=0.0,
        optimizer='rmsprop', loss='cross_entropy',
        num_layers=3, hidden_size=128,
        activation='relu', weight_init='xavier'
    )
    defaults.update(kwargs)
    return types.SimpleNamespace(**defaults)


def train_model(args, X_tr, y_tr, X_val, y_val, run=None, extra_hooks=None):
    model = NeuralNetwork(args)
    loss_fn, _ = get_loss(args.loss)
    n = X_tr.shape[0]
    history = []
    for epoch in range(args.epochs):
        idx = np.random.permutation(n)
        Xs, ys = X_tr[idx], y_tr[idx]
        ep_loss, nb = 0.0, 0
        for s in range(0, n, args.batch_size):
            Xb = Xs[s:s+args.batch_size]; yb = ys[s:s+args.batch_size]
            fwd = model.forward(Xb)
            ep_loss += loss_fn(yb, fwd); nb += 1
            model.backward(yb, fwd); model.update_weights()
        avg_loss = ep_loss / nb
        tr_acc   = model.evaluate(X_tr, y_tr)
        val_log  = model.forward(X_val)
        val_loss = float(loss_fn(y_val, val_log))
        val_acc  = model.evaluate(X_val, y_val)
        rec = dict(epoch=epoch+1, train_loss=avg_loss, train_accuracy=tr_acc,
                   val_loss=val_loss, val_accuracy=val_acc)
        history.append(rec)
        if run: run.log(rec)
        if extra_hooks: extra_hooks(model, epoch, run)
        print(f'  Ep {epoch+1:>3}/{args.epochs}  loss={avg_loss:.4f}  tr={tr_acc:.4f}  val={val_acc:.4f}')
    return model, history


def plot_cm(cm, class_names, title='Confusion Matrix', cmap='Blues', figsize=(10,8)):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    thresh = cm.max() / 2.0
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=7,
                    color='white' if cm[i,j] > thresh else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.tight_layout()
    return fig

print('✅  Helpers defined')

✅  Helpers defined


## 📦 Load Datasets

In [5]:
print('Loading MNIST ...')
Xtr_m,Xv_m,Xte_m,ytr_m,yv_m,yte_m,yte_m_raw = load_data('mnist')
print('Loading Fashion-MNIST ...')
Xtr_f,Xv_f,Xte_f,ytr_f,yv_f,yte_f,yte_f_raw = load_data('fashion_mnist')
MNIST_CLASSES   = [str(i) for i in range(10)]
FASHION_CLASSES = ['T-shirt','Trouser','Pullover','Dress','Coat',
                   'Sandal','Shirt','Sneaker','Bag','Ankle boot']
print('✅  Datasets loaded')

Loading MNIST ...
Loaded mnist:
  Train: (54000, 784), Val: (6000, 784), Test: (10000, 784)
Loading Fashion-MNIST ...
Loaded fashion_mnist:
  Train: (54000, 784), Val: (6000, 784), Test: (10000, 784)
✅  Datasets loaded


## 2.1 Data Exploration and Class Distribution

In [ ]:

from keras.datasets import mnist as _mnist
(raw_X_m, raw_y_m), _ = _mnist.load_data()   # (60000, 28, 28) uint8

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q1_data_exploration_mnist', reinit=True)

# 1. W&B Table: 5 samples × 10 classes 
table = wandb.Table(columns=['class_id', 'class_name', 'image', 'sample_index'])
for cls in range(10):
    for idx in np.where(raw_y_m == cls)[0][:5]:
        table.add_data(cls, str(cls), wandb.Image(raw_X_m[idx]), int(idx))
run.log({'class_samples_table': table})

# 2. Class distribution bar chart 
counts = [int(np.sum(raw_y_m == c)) for c in range(10)]
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(MNIST_CLASSES, counts, color=plt.cm.tab10.colors)
ax.set_title('MNIST — Class Distribution (Training Set)', fontsize=13)
ax.set_ylabel('Sample Count')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(cnt), ha='center', fontsize=8)
plt.tight_layout()
run.log({'class_distribution': wandb.Image(fig)}); plt.close(fig)

# 3. 10×5 sample grid 
fig, axes = plt.subplots(10, 5, figsize=(10, 21))
for cls in range(10):
    idxs = np.where(raw_y_m == cls)[0][:5]
    for j, idx in enumerate(idxs):
        axes[cls][j].imshow(raw_X_m[idx], cmap='gray')
        axes[cls][j].axis('off')
    axes[cls][0].set_ylabel(f'Digit {cls}', rotation=0,
                             ha='right', labelpad=45, fontsize=9)
plt.suptitle('5 Samples per Class — MNIST', fontsize=14, y=1.01)
plt.tight_layout()
run.log({'sample_grid': wandb.Image(fig)}); plt.close(fig)

run.finish()
print('Q2.1 MNIST done')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Q2.1 MNIST done


## 2.2 Hyperparameter Sweep

In [ ]:

sweep_config_mnist = {
    'method': 'bayes',
    'metric': {'name': 'val_accuracy', 'goal': 'maximize'},
    'parameters': {
        'learning_rate': {'values': [0.0001, 0.0005, 0.001, 0.005, 0.01]},
        'optimizer':     {'values': ['sgd', 'momentum', 'nag', 'rmsprop']},
        'num_layers':    {'values': [2, 3, 4, 5]},
        'hidden_size':   {'values': [32, 64, 128]},
        'activation':    {'values': ['relu', 'sigmoid', 'tanh']},
        'weight_decay':  {'values': [0.0, 0.0001, 0.0005, 0.001]},
        'batch_size':    {'values': [32, 64, 128]},
        'weight_init':   {'values': ['random', 'xavier']},
        'loss':          {'values': ['cross_entropy', 'mse']},
        'epochs':        {'value': 10},
    }
}


def _sweep_train_mnist():
    run = wandb.init()
    c = run.config
    args = make_args(
        dataset='mnist',
        epochs=c.epochs,          batch_size=c.batch_size,
        learning_rate=c.learning_rate, weight_decay=c.weight_decay,
        optimizer=c.optimizer,    loss=c.loss,
        num_layers=c.num_layers,  hidden_size=c.hidden_size,
        activation=c.activation,  weight_init=c.weight_init
    )
    model, _ = train_model(args, Xtr_m, ytr_m, Xv_m, yv_m, run=run)
    run.log({'test_accuracy': model.evaluate(Xte_m, yte_m)})
    run.finish()


sweep_id_mnist = wandb.sweep(sweep_config_mnist,
                              project=WANDB_PROJECT, entity=WANDB_ENTITY)
print(f'MNIST Sweep ID: {sweep_id_mnist} — starting 100 runs ...')
wandb.agent(sweep_id_mnist, _sweep_train_mnist, count=100)
print('Q2.2 MNIST sweep complete')

Create sweep with ID: 2up4yugq
Sweep URL: https://wandb.ai/da25m024-iitm/da6401_assignment1/sweeps/2up4yugq
MNIST Sweep ID: 2up4yugq — starting 100 runs ...


wandb: Agent Starting Run: nqrtf8sb with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.0001
wandb: 	loss: cross_entropy
wandb: 	num_layers: 2
wandb: 	optimizer: momentum
wandb: 	weight_decay: 0.0005
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\theri\_netrc.


Built network: 2 hidden layers, 128 neurons, activation = sigmoid
  Ep   1/10  loss=2.3184  tr=0.1143  val=0.1153
  Ep   2/10  loss=2.2782  tr=0.1376  val=0.1357
  Ep   3/10  loss=2.2617  tr=0.2248  val=0.2215
  Ep   4/10  loss=2.2443  tr=0.2682  val=0.2682
  Ep   5/10  loss=2.2255  tr=0.3946  val=0.3935
  Ep   6/10  loss=2.2049  tr=0.3954  val=0.3953
  Ep   7/10  loss=2.1819  tr=0.5089  val=0.5072
  Ep   8/10  loss=2.1557  tr=0.4896  val=0.4903
  Ep   9/10  loss=2.1260  tr=0.5234  val=0.5213
  Ep  10/10  loss=2.0917  tr=0.5661  val=0.5648


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▁▃▃▅▅▇▇▇█
train_loss,█▇▆▆▅▄▄▃▂▁
val_accuracy,▁▁▃▃▅▅▇▇▇█
val_loss,█▇▇▆▆▅▄▃▂▁
epoch,10
test_accuracy,0.5746
train_accuracy,0.56615
train_loss,2.09168
val_accuracy,0.56483


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: nvug1glg with config:
wandb: 	activation: relu
wandb: 	batch_size: 128
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.0005
wandb: 	loss: mse
wandb: 	num_layers: 5
wandb: 	optimizer: momentum
wandb: 	weight_decay: 0.001
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\theri\_netrc.


...

epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▄▅█▆▃▆▇██
train_loss,█▃▂▂▁▁▁▁▁▁
val_accuracy,▁▃▅█▆▃▆▇██
val_loss,█▆▄▁▃▆▃▂▁▁
epoch,10
test_accuracy,0.9668
train_accuracy,0.97089
train_loss,0.12406
val_accuracy,0.96583


wandb: Agent Starting Run: o1s5ktqm with config:
wandb: 	activation: relu
wandb: 	batch_size: 64
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.01
wandb: 	loss: cross_entropy
wandb: 	num_layers: 2
wandb: 	optimizer: rmsprop
wandb: 	weight_decay: 0.001
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\theri\_netrc.


Built network: 2 hidden layers, 128 neurons, activation = relu
  Ep   1/10  loss=0.3482  tr=0.9510  val=0.9468
  Ep   2/10  loss=0.2316  tr=0.8947  val=0.8943
  Ep   3/10  loss=0.2179  tr=0.9248  val=0.9222
  Ep   4/10  loss=0.2098  tr=0.9456  val=0.9412
  Ep   5/10  loss=0.2080  tr=0.9196  val=0.9135
  Ep   6/10  loss=0.2074  tr=0.9529  val=0.9483
  Ep   7/10  loss=0.2025  tr=0.9425  val=0.9388
  Ep   8/10  loss=0.2073  tr=0.9228  val=0.9147
  Ep   9/10  loss=0.2015  tr=0.9181  val=0.9143
  Ep  10/10  loss=0.2034  tr=0.9196  val=0.9202


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,█▁▅▇▄█▇▄▄▄
train_loss,█▂▂▁▁▁▁▁▁▁
val_accuracy,█▁▅▇▃█▇▄▄▄
val_loss,▁█▅▂▅▁▂▅▅▅
epoch,10
test_accuracy,0.916
train_accuracy,0.91959
train_loss,0.20345
val_accuracy,0.92017


Q2.2 MNIST sweep complete


In [9]:
OPTIMIZERS  = ['sgd','momentum','nag','rmsprop']
OPT_COLORS  = {'sgd':'#e41a1c','momentum':'#377eb8','nag':'#4daf4a','rmsprop':'#984ea3'}
opt_hist    = {}

for opt in OPTIMIZERS:
    print(f'\n▶  {opt}')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=f'q3_optimizer_{opt}', group='q3_optimizer_showdown',
                     config={'optimizer':opt,'num_layers':3,'hidden_size':128,
                             'activation':'relu','learning_rate':0.001,'epochs':10},
                     reinit=True)
    args = make_args(optimizer=opt, num_layers=3, hidden_size=128,
                     activation='relu', learning_rate=0.001, epochs=10)
    model, hist = train_model(args, Xtr_m, ytr_m, Xv_m, yv_m, run=run)
    opt_hist[opt] = hist
    run.log({'test_accuracy': model.evaluate(Xte_m, yte_m)})
    run.finish()

# Overlay comparison plot
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q3_comparison_overlay', reinit=True)
fig, axes = plt.subplots(1,2, figsize=(14,5))
for opt in OPTIMIZERS:
    ep=[h['epoch'] for h in opt_hist[opt]]; c=OPT_COLORS[opt]
    axes[0].plot(ep,[h['train_loss'] for h in opt_hist[opt]], label=opt, color=c, lw=2, marker='o', ms=4)
    axes[1].plot(ep,[h['val_accuracy'] for h in opt_hist[opt]], label=opt, color=c, lw=2, marker='o', ms=4)
for ax,(t,y) in zip(axes,[('Training Loss','Loss'),('Validation Accuracy','Accuracy')]):
    ax.set_title(t); ax.set_xlabel('Epoch'); ax.set_ylabel(y); ax.legend(); ax.grid(True,alpha=0.3)
plt.suptitle('Q2.3 — Optimizer Showdown  (3 layers · 128 neurons · ReLU · LR=0.001)', fontsize=12)
plt.tight_layout()
run.log({'optimizer_comparison': wandb.Image(fig)}); plt.close(fig)
run.finish()
print('Q2.3 done')


▶  sgd


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/10  loss=2.1920  tr=0.4752  val=0.4633
  Ep   2/10  loss=1.8991  tr=0.6581  val=0.6495
  Ep   3/10  loss=1.4256  tr=0.7567  val=0.7512
  Ep   4/10  loss=0.9961  tr=0.8036  val=0.8042
  Ep   5/10  loss=0.7635  tr=0.8275  val=0.8265
  Ep   6/10  loss=0.6405  tr=0.8451  val=0.8437
  Ep   7/10  loss=0.5649  tr=0.8577  val=0.8573
  Ep   8/10  loss=0.5131  tr=0.8656  val=0.8638
  Ep   9/10  loss=0.4752  tr=0.8761  val=0.8750
  Ep  10/10  loss=0.4457  tr=0.8816  val=0.8778


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▄▆▇▇▇████
train_loss,█▇▅▃▂▂▁▁▁▁
val_accuracy,▁▄▆▇▇▇████
val_loss,█▆▄▃▂▂▁▁▁▁
epoch,10
test_accuracy,0.8882
train_accuracy,0.88159
train_loss,0.44567
val_accuracy,0.87783



▶  momentum


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/10  loss=1.0279  tr=0.8804  val=0.8753
  Ep   2/10  loss=0.3606  tr=0.9076  val=0.9032
  Ep   3/10  loss=0.2936  tr=0.9229  val=0.9218
  Ep   4/10  loss=0.2551  tr=0.9309  val=0.9252
  Ep   5/10  loss=0.2268  tr=0.9394  val=0.9343
  Ep   6/10  loss=0.2047  tr=0.9441  val=0.9380
  Ep   7/10  loss=0.1862  tr=0.9494  val=0.9432
  Ep   8/10  loss=0.1705  tr=0.9541  val=0.9470
  Ep   9/10  loss=0.1570  tr=0.9572  val=0.9492
  Ep  10/10  loss=0.1451  tr=0.9607  val=0.9542


...

epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▄▅▅▆▇▇▇██
train_loss,█▃▂▂▂▁▁▁▁▁
val_accuracy,▁▄▅▅▆▇▇▇██
val_loss,█▅▄▄▃▂▂▂▁▁
epoch,10
test_accuracy,0.9554
train_accuracy,0.96135
train_loss,0.14571
val_accuracy,0.95467



▶  rmsprop


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/10  loss=0.2680  tr=0.9649  val=0.9545
  Ep   2/10  loss=0.1127  tr=0.9764  val=0.9655
  Ep   3/10  loss=0.0801  tr=0.9719  val=0.9572
  Ep   4/10  loss=0.0605  tr=0.9887  val=0.9737
  Ep   5/10  loss=0.0497  tr=0.9893  val=0.9732
  Ep   6/10  loss=0.0413  tr=0.9925  val=0.9748
  Ep   7/10  loss=0.0356  tr=0.9945  val=0.9758
  Ep   8/10  loss=0.0302  tr=0.9927  val=0.9705
  Ep   9/10  loss=0.0260  tr=0.9950  val=0.9752
  Ep  10/10  loss=0.0224  tr=0.9938  val=0.9712


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▄▃▇▇▇█▇██
train_loss,█▄▃▂▂▂▁▁▁▁
val_accuracy,▁▅▂▇▇██▆█▆
val_loss,▇▃▆▁▂▂▂▅▄█
epoch,10
test_accuracy,0.9751
train_accuracy,0.99378
train_loss,0.02243
val_accuracy,0.97117


Q2.3 done


## Re-log Q2.3 overlay with a unique name to avoid conflicts

In [10]:
# Re-log Q2.3 overlay with a unique name to avoid conflicts
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q3_comparison_overlay_v2', reinit=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
OPT_COLORS = {'sgd':'#e41a1c','momentum':'#377eb8','nag':'#4daf4a','rmsprop':'#984ea3'}

for opt in ['sgd','momentum','nag','rmsprop']:
    ep  = [h['epoch']        for h in opt_hist[opt]]
    c   = OPT_COLORS[opt]
    axes[0].plot(ep, [h['train_loss']    for h in opt_hist[opt]], label=opt, color=c, lw=2, marker='o', ms=4)
    axes[1].plot(ep, [h['val_accuracy']  for h in opt_hist[opt]], label=opt, color=c, lw=2, marker='o', ms=4)

for ax, (t, y) in zip(axes, [('Training Loss','Loss'),('Validation Accuracy','Accuracy')]):
    ax.set_title(t); ax.set_xlabel('Epoch'); ax.set_ylabel(y)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Q2.3 — Optimizer Showdown (3 layers · 128 neurons · ReLU · LR=0.001)', fontsize=12)
plt.tight_layout()
run.log({'optimizer_comparison': wandb.Image(fig)})
plt.close(fig)
run.finish()
print('✅ Q2.3 overlay re-logged as q3_comparison_overlay_v2')

✅ Q2.3 overlay re-logged as q3_comparison_overlay_v2


## 2.3 The Optimizer Showdown

In [11]:
# Complete Q2.3 fix — trains all 4 optimizers AND logs overlay in one cell
import wandb, numpy as np, matplotlib.pyplot as plt, types

OPT_COLORS = {'sgd':'#e41a1c','momentum':'#377eb8',
              'nag':'#4daf4a','rmsprop':'#984ea3'}
opt_hist_fix = {}

for opt in ['sgd','momentum','nag','rmsprop']:
    print(f'\n▶  {opt}')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=f'q3v2_optimizer_{opt}',
                     group='q3v2_optimizer_showdown', reinit=True)
    args = make_args(optimizer=opt, num_layers=3, hidden_size=128,
                     activation='relu', learning_rate=0.001, epochs=10)
    model, hist = train_model(args, Xtr_m, ytr_m, Xv_m, yv_m, run=run)
    opt_hist_fix[opt] = hist
    run.log({'test_accuracy': model.evaluate(Xte_m, yte_m)})
    run.finish()

# Now log overlay immediately while opt_hist_fix is in memory
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q3v2_comparison_overlay', reinit=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt in ['sgd','momentum','nag','rmsprop']:
    ep = [h['epoch'] for h in opt_hist_fix[opt]]
    c  = OPT_COLORS[opt]
    axes[0].plot(ep, [h['train_loss']   for h in opt_hist_fix[opt]], label=opt, color=c, lw=2, marker='o', ms=4)
    axes[1].plot(ep, [h['val_accuracy'] for h in opt_hist_fix[opt]], label=opt, color=c, lw=2, marker='o', ms=4)
for ax, (t, y) in zip(axes, [('Training Loss','Loss'),('Validation Accuracy','Accuracy')]):
    ax.set_title(t); ax.set_xlabel('Epoch')
    ax.set_ylabel(y); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Q2.3 — Optimizer Showdown', fontsize=12)
plt.tight_layout()
run.log({'optimizer_comparison': wandb.Image(fig)})
plt.close(fig)
run.finish()
print('✅  Q2.3 fully fixed — run: q3v2_comparison_overlay')


▶  sgd


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/10  loss=2.2476  tr=0.3738  val=0.3730
  Ep   2/10  loss=2.0553  tr=0.6286  val=0.6265
  Ep   3/10  loss=1.6633  tr=0.7286  val=0.7197
  Ep   4/10  loss=1.1633  tr=0.7871  val=0.7812
  Ep   5/10  loss=0.8399  tr=0.8164  val=0.8123
  Ep   6/10  loss=0.6759  tr=0.8368  val=0.8365
  Ep   7/10  loss=0.5836  tr=0.8518  val=0.8483
  Ep   8/10  loss=0.5240  tr=0.8628  val=0.8592
  Ep   9/10  loss=0.4812  tr=0.8726  val=0.8702
  Ep  10/10  loss=0.4491  tr=0.8789  val=0.8760


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▅▆▇▇▇████
train_loss,█▇▆▄▃▂▂▁▁▁
val_accuracy,▁▅▆▇▇▇████
val_loss,█▇▅▃▂▂▁▁▁▁
epoch,10
test_accuracy,0.8887
train_accuracy,0.87887
train_loss,0.44913
val_accuracy,0.876



▶  momentum


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/10  loss=1.0610  tr=0.8772  val=0.8743
  Ep   2/10  loss=0.3642  tr=0.9085  val=0.9065
  Ep   3/10  loss=0.2916  tr=0.9218  val=0.9205
  Ep   4/10  loss=0.2546  tr=0.9312  val=0.9293
  Ep   5/10  loss=0.2267  tr=0.9397  val=0.9363
  Ep   6/10  loss=0.2055  tr=0.9443  val=0.9405
  Ep   7/10  loss=0.1873  tr=0.9481  val=0.9432
  Ep   8/10  loss=0.1731  tr=0.9526  val=0.9477
  Ep   9/10  loss=0.1595  tr=0.9561  val=0.9497
  Ep  10/10  loss=0.1488  tr=0.9591  val=0.9528


...

epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▄▅▆▆▇▇███
train_loss,█▃▂▂▂▁▁▁▁▁
val_accuracy,▁▄▅▆▆▇▇▇██
val_loss,█▅▄▃▃▂▂▁▁▁
epoch,10
test_accuracy,0.9537
train_accuracy,0.95757
train_loss,0.15386
val_accuracy,0.9505



▶  rmsprop


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/10  loss=0.2671  tr=0.9659  val=0.9603
  Ep   2/10  loss=0.1103  tr=0.9759  val=0.9672
  Ep   3/10  loss=0.0797  tr=0.9820  val=0.9705
  Ep   4/10  loss=0.0619  tr=0.9856  val=0.9713
  Ep   5/10  loss=0.0505  tr=0.9905  val=0.9778
  Ep   6/10  loss=0.0405  tr=0.9913  val=0.9755
  Ep   7/10  loss=0.0353  tr=0.9923  val=0.9730
  Ep   8/10  loss=0.0299  tr=0.9930  val=0.9768
  Ep   9/10  loss=0.0248  tr=0.9966  val=0.9767
  Ep  10/10  loss=0.0219  tr=0.9940  val=0.9738


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▃▅▆▇▇▇▇█▇
train_loss,█▄▃▂▂▂▁▁▁▁
val_accuracy,▁▄▅▅█▇▆██▆
val_loss,▇▄▂▄▁▂▆▆▇█
epoch,10
test_accuracy,0.9763
train_accuracy,0.99402
train_loss,0.02189
val_accuracy,0.97383


✅  Q2.3 fully fixed — run: q3v2_comparison_overlay


In [ ]:

import wandb, matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

unique_name = f'q3_overlay_final_{datetime.now().strftime("%H%M%S")}'

OPT_COLORS = {'sgd':'#e41a1c','momentum':'#377eb8',
              'nag':'#4daf4a','rmsprop':'#984ea3'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt in ['sgd','momentum','nag','rmsprop']:
    ep = [h['epoch'] for h in opt_hist_fix[opt]]
    c  = OPT_COLORS[opt]
    axes[0].plot(ep, [h['train_loss']   for h in opt_hist_fix[opt]], label=opt, color=c, lw=2, marker='o', ms=4)
    axes[1].plot(ep, [h['val_accuracy'] for h in opt_hist_fix[opt]], label=opt, color=c, lw=2, marker='o', ms=4)
for ax, (t, y) in zip(axes, [('Training Loss','Loss'),('Validation Accuracy','Accuracy')]):
    ax.set_title(t); ax.set_xlabel('Epoch')
    ax.set_ylabel(y); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Q2.3 — Optimizer Showdown', fontsize=12)
plt.tight_layout()

# Save image locally first
img_path = 'q3_optimizer_comparison.png'
plt.savefig(img_path, dpi=150, bbox_inches='tight')
plt.close(fig)

# Log to completely fresh run
wandb.finish()  # kill any existing run
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name=unique_name,
                 settings=wandb.Settings(start_method="thread"),
                 reinit=True)
run.log({'optimizer_comparison': wandb.Image(img_path)})
run.finish()
print(f'✅ Logged to run: {unique_name}')
print(f'   Also saved locally as: {img_path}')

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


✅ Logged to run: q3_overlay_final_163127
   Also saved locally as: q3_optimizer_comparison.png


In [ ]:

import wandb

api = wandb.Api()

# List of run names that are corrupted
old_runs = [
    'q4_grad_norm_overlay',
    'q4_sigmoid_L3', 'q4_sigmoid_L5',
    'q4_relu_L3', 'q4_relu_L5',
]

for run_name in old_runs:
    try:
        runs = api.runs(f"da25m024-iitm/da6401_assignment1", 
                       filters={"display_name": run_name})
        for run in runs:
            print(f"Deleting: {run.name} (id: {run.id})")
            run.delete()
    except Exception as e:
        print(f"Could not delete {run_name}: {e}")

print("✅ Done — now re-run the Q2.4 cell")

✅ Done — now re-run the Q2.4 cell


---
## Q2.4 — Vanishing Gradient Analysis

In [ ]:
CONFIGS_Q4 = [
    {'activation':'sigmoid','num_layers':3},
    {'activation':'sigmoid','num_layers':5},
    {'activation':'relu',   'num_layers':3},
    {'activation':'relu',   'num_layers':5},
]
q4_data = {}

for cfg in CONFIGS_Q4:
    act, nl = cfg['activation'], cfg['num_layers']
    rname = f'q4_{act}_L{nl}'
    print(f'\n▶  {rname}')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=rname, group='q4_vanishing_gradient',
                     config={'activation':act,'num_layers':nl,'optimizer':'rmsprop'},
                     reinit=True)
    args = make_args(optimizer='rmsprop', activation=act,
                     num_layers=nl, hidden_size=128, learning_rate=0.001, epochs=15)
    gn_list = []
    def _hook(model, epoch, run, _l=gn_list):
        gn = float(np.linalg.norm(model.layers[0].grad_W))
        _l.append(gn)
        if run: run.log({'epoch':epoch+1,'first_layer_grad_norm':gn})
    train_model(args, Xtr_m, ytr_m, Xv_m, yv_m, run=run, extra_hooks=_hook)
    q4_data[rname] = list(gn_list)
    run.finish()

# Overlay
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q4_grad_norm_overlay', reinit=True)
fig, ax = plt.subplots(figsize=(10,5))
smap = {'sigmoid_L3':('--','#e41a1c'),'sigmoid_L5':('--','#ff7f00'),
        'relu_L3':('-','#377eb8'),'relu_L5':('-','#4daf4a')}
for rname, norms in q4_data.items():
    key = ('sigmoid' if 'sigmoid' in rname else 'relu') + ('_L3' if 'L3' in rname else '_L5')
    ls, col = smap[key]
    ax.plot(range(1,len(norms)+1), norms, label=rname, linestyle=ls, color=col, lw=2)
ax.set_title('Q2.4 — First Hidden-Layer Gradient Norm  (RMSProp · MNIST · 15 epochs)')
ax.set_xlabel('Epoch'); ax.set_ylabel('||∇W||'); ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout()
run.log({'vanishing_gradient_overlay': wandb.Image(fig)}); plt.close(fig)
run.finish()
print('✅  Q2.4 done')


▶  q4_sigmoid_L3


KeyboardInterrupt: 

---
## Q2.5 — Dead Neuron Investigation

In [4]:


CONFIGS_Q5 = [
    {'activation':'relu', 'learning_rate':0.1,   'label':'relu_highLR'},
    {'activation':'relu', 'learning_rate':0.001, 'label':'relu_normalLR'},
    {'activation':'tanh', 'learning_rate':0.1,   'label':'tanh_highLR'},
]
q5_data = {}

for cfg in CONFIGS_Q5:
    label = cfg['label']
    print(f'\n▶  {label}')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=f'q5_{label}', group='q5_dead_neurons',
                     config=cfg, reinit=True)

    args = make_args(activation=cfg['activation'],
                     learning_rate=cfg['learning_rate'],
                     num_layers=3, hidden_size=128, epochs=15)

    dead_ep = []; grad_ep = []

    def _hook5(model, epoch, run, _d=dead_ep, _g=grad_ep):
        _ = model.forward(Xtr_m[:512])
        act0 = model.layers[0].a_out          # (512, 128)

        # ── guard: replace any NaN/Inf in activations ──────────────
        has_nan = not np.all(np.isfinite(act0))
        act0_safe = np.nan_to_num(act0, nan=0.0, posinf=0.0, neginf=0.0)

        # dead neuron: fires 0 for every sample in the batch
        dead = float(np.mean(np.all(act0_safe == 0, axis=0)) * 100)

        # grad_W can also be NaN if forward exploded
        gw = model.layers[0].grad_W
        gw_safe = np.nan_to_num(gw, nan=0.0, posinf=0.0, neginf=0.0)
        gn = float(np.linalg.norm(gw_safe))

        _d.append(dead); _g.append(gn)

        if run:
            log_dict = {
                'epoch':            epoch + 1,
                'dead_neuron_pct':  dead,
                'layer0_grad_norm': gn,
                'has_nan_activation': float(has_nan),
            }
            # only log histogram when values are finite
            flat = act0_safe.flatten()
            if np.isfinite(flat).all() and flat.max() != flat.min():
                log_dict['activation_histogram'] = wandb.Histogram(
                    flat, num_bins=64)
            run.log(log_dict)

    model, hist = train_model(args, Xtr_m, ytr_m, Xv_m, yv_m,
                               run=run, extra_hooks=_hook5)
    q5_data[label] = {
        'dead':    list(dead_ep),
        'grad':    list(grad_ep),
        'val_acc': [h['val_accuracy'] for h in hist],
    }
    run.finish()

# ── summary overlay plot ──────────────────────────────────────────────────────
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q5_dead_neuron_summary', reinit=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pal = ['#e41a1c', '#377eb8', '#4daf4a']

for i, (label, data) in enumerate(q5_data.items()):
    ep = range(1, len(data['dead']) + 1)
    axes[0].plot(ep, data['dead'],    label=label, color=pal[i], lw=2)
    axes[1].plot(ep, data['grad'],    label=label, color=pal[i], lw=2)
    axes[2].plot(ep, data['val_acc'], label=label, color=pal[i], lw=2)

for ax, (t, y) in zip(axes, [
        ('Dead Neuron % (Layer 0)', '%'),
        ('Layer-0 Gradient Norm',   '||∇W||'),
        ('Validation Accuracy',     'Acc')]):
    ax.set_title(t); ax.set_xlabel('Epoch')
    ax.set_ylabel(y); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Q2.5 — Dead Neuron Investigation', fontsize=13)
plt.tight_layout()
run.log({'dead_neuron_summary': wandb.Image(fig)}); plt.close(fig)
run.finish()
print('Q2.5 done')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\theri\_netrc.



▶  relu_highLR


wandb: Currently logged in as: da25m024 (da25m024-iitm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Built network: 3 hidden layers, 128 neurons, activation = relu


c:\Users\theri\Desktop\sem 2\DL\assignment_1\src\ann\objective_functions.py:15: RuntimeWarning: overflow encountered in exp
  exp_z = np.exp(z)
c:\Users\theri\Desktop\sem 2\DL\assignment_1\src\ann\objective_functions.py:16: RuntimeWarning: invalid value encountered in divide
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)


...

dead_neuron_pct,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
has_nan_activation,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
layer0_grad_norm,▃▂▂▃▂▂▄▂▁█▁▂▄▂▄
train_accuracy,▁▄▅▅▆▇▇▇████▇██
train_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_accuracy,▁▄▆▄▆▆▅▅█▇█▇▆▇█
val_loss,▄▁▁▂▁▂▃▄▂▃▄▄█▅▅
dead_neuron_pct,0
epoch,15
has_nan_activation,0



▶  tanh_highLR


Built network: 3 hidden layers, 128 neurons, activation = tanh
  Ep   1/15  loss=14.5857  tr=0.1124  val=0.1123
  Ep   2/15  loss=nan  tr=0.0987  val=0.0987
  Ep   3/15  loss=nan  tr=0.0987  val=0.0987
  Ep   4/15  loss=nan  tr=0.0987  val=0.0987
  Ep   5/15  loss=nan  tr=0.0987  val=0.0987
  Ep   6/15  loss=nan  tr=0.0987  val=0.0987
  Ep   7/15  loss=nan  tr=0.0987  val=0.0987
  Ep   8/15  loss=nan  tr=0.0987  val=0.0987
  Ep   9/15  loss=nan  tr=0.0987  val=0.0987
  Ep  10/15  loss=nan  tr=0.0987  val=0.0987
  Ep  11/15  loss=nan  tr=0.0987  val=0.0987
  Ep  12/15  loss=nan  tr=0.0987  val=0.0987
  Ep  13/15  loss=nan  tr=0.0987  val=0.0987
  Ep  14/15  loss=nan  tr=0.0987  val=0.0987
  Ep  15/15  loss=nan  tr=0.0987  val=0.0987


dead_neuron_pct,▁██████████████
epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
has_nan_activation,▁██████████████
layer0_grad_norm,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,▁
val_accuracy,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁
dead_neuron_pct,100
epoch,15
has_nan_activation,1


Q2.5 done


---
## Q2.6 — Loss Function Comparison

In [5]:
LOSSES = ['cross_entropy','mse']
LCOLS  = {'cross_entropy':'#377eb8','mse':'#e41a1c'}
q6_hist = {}

for ln in LOSSES:
    print(f'\n▶  {ln}')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=f'q6_loss_{ln}', group='q6_loss_comparison',
                     config={'loss':ln,'optimizer':'rmsprop','num_layers':3,
                             'hidden_size':128,'activation':'relu','learning_rate':0.001},
                     reinit=True)
    args = make_args(loss=ln, optimizer='rmsprop', num_layers=3, hidden_size=128,
                     activation='relu', learning_rate=0.001, epochs=15)
    model, hist = train_model(args, Xtr_m, ytr_m, Xv_m, yv_m, run=run)
    q6_hist[ln] = hist
    run.log({'test_accuracy': model.evaluate(Xte_m, yte_m)})
    run.finish()

# Overlay plot
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q6_loss_comparison_plot', reinit=True)
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ln,hist in q6_hist.items():
    ep=[h['epoch'] for h in hist]; c=LCOLS[ln]
    axes[0].plot(ep,[h['train_loss'] for h in hist], label=ln, color=c, lw=2)
    axes[1].plot(ep,[h['val_accuracy'] for h in hist], label=ln, color=c, lw=2)
for ax,(t,y) in zip(axes,[('Training Loss','Loss'),('Validation Accuracy','Accuracy')]):
    ax.set_title(t); ax.set_xlabel('Epoch'); ax.set_ylabel(y); ax.legend(); ax.grid(True,alpha=0.3)
plt.suptitle('Q2.6 — MSE vs Cross-Entropy  (same arch · same LR · RMSProp)', fontsize=12)
plt.tight_layout()
run.log({'loss_comparison': wandb.Image(fig)}); plt.close(fig)
run.finish()
print('Q2.6 done')


▶  cross_entropy


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/15  loss=0.2615  tr=0.9518  val=0.9465
  Ep   2/15  loss=0.1112  tr=0.9812  val=0.9695
  Ep   3/15  loss=0.0798  tr=0.9837  val=0.9723
  Ep   4/15  loss=0.0594  tr=0.9885  val=0.9723
  Ep   5/15  loss=0.0493  tr=0.9875  val=0.9698
  Ep   6/15  loss=0.0415  tr=0.9926  val=0.9755
  Ep   7/15  loss=0.0348  tr=0.9933  val=0.9732
  Ep   8/15  loss=0.0289  tr=0.9951  val=0.9760
  Ep   9/15  loss=0.0266  tr=0.9944  val=0.9738
  Ep  10/15  loss=0.0219  tr=0.9947  val=0.9745
  Ep  11/15  loss=0.0203  tr=0.9932  val=0.9732
  Ep  12/15  loss=0.0159  tr=0.9970  val=0.9753
  Ep  13/15  loss=0.0181  tr=0.9970  val=0.9760
  Ep  14/15  loss=0.0146  tr=0.9950  val=0.9745
  Ep  15/15  loss=0.0148  tr=0.9962  val=0.9748


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▁
train_accuracy,▁▆▆▇▇▇▇███▇████
train_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_accuracy,▁▆▇▇▇█▇█▇█▇████
val_loss,▅▁▁▁▂▁▂▂▃▄▄▅▅█▆
epoch,15
test_accuracy,0.9756
train_accuracy,0.99622
train_loss,0.01482
val_accuracy,0.97483



▶  mse


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/15  loss=0.1259  tr=0.9621  val=0.9583
  Ep   2/15  loss=0.0596  tr=0.9729  val=0.9658
  Ep   3/15  loss=0.0455  tr=0.9797  val=0.9650
  Ep   4/15  loss=0.0367  tr=0.9794  val=0.9655
  Ep   5/15  loss=0.0306  tr=0.9834  val=0.9707
  Ep   6/15  loss=0.0270  tr=0.9841  val=0.9680
  Ep   7/15  loss=0.0240  tr=0.9880  val=0.9735
  Ep   8/15  loss=0.0211  tr=0.9894  val=0.9737
  Ep   9/15  loss=0.0194  tr=0.9901  val=0.9727
  Ep  10/15  loss=0.0179  tr=0.9908  val=0.9702
  Ep  11/15  loss=0.0164  tr=0.9916  val=0.9727
  Ep  12/15  loss=0.0139  tr=0.9908  val=0.9695
  Ep  13/15  loss=0.0140  tr=0.9924  val=0.9743
  Ep  14/15  loss=0.0124  tr=0.9924  val=0.9733
  Ep  15/15  loss=0.0121  tr=0.9949  val=0.9755


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▁
train_accuracy,▁▃▅▅▆▆▇▇▇▇▇▇▇▇█
train_loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val_accuracy,▁▄▄▄▆▅▇▇▇▆▇▆█▇█
val_loss,█▅▅▄▂▄▁▂▂▃▂▄▂▂▁
epoch,15
test_accuracy,0.9785
train_accuracy,0.99494
train_loss,0.01208
val_accuracy,0.9755


Q2.6 done


---
## Q2.7 — Global Performance Analysis (12 configs)

In [6]:
CFG_Q7 = [
    make_args(optimizer='sgd',      num_layers=2,hidden_size=64, activation='relu',    epochs=10,learning_rate=0.01),
    make_args(optimizer='sgd',      num_layers=4,hidden_size=128,activation='sigmoid', epochs=10,learning_rate=0.01),
    make_args(optimizer='momentum', num_layers=3,hidden_size=128,activation='relu',    epochs=10,learning_rate=0.001),
    make_args(optimizer='momentum', num_layers=5,hidden_size=128,activation='tanh',    epochs=10,learning_rate=0.005),
    make_args(optimizer='nag',      num_layers=3,hidden_size=64, activation='relu',    epochs=10,learning_rate=0.001),
    make_args(optimizer='nag',      num_layers=4,hidden_size=128,activation='tanh',    epochs=10,learning_rate=0.001),
    make_args(optimizer='rmsprop',  num_layers=3,hidden_size=128,activation='relu',    epochs=10,learning_rate=0.001),
    make_args(optimizer='rmsprop',  num_layers=3,hidden_size=128,activation='relu',    epochs=10,learning_rate=0.001,weight_decay=0.001),
    make_args(optimizer='rmsprop',  num_layers=5,hidden_size=128,activation='relu',    epochs=10,learning_rate=0.001),
    make_args(optimizer='rmsprop',  num_layers=2,hidden_size=32, activation='sigmoid', epochs=10,learning_rate=0.001),
    make_args(optimizer='rmsprop',  num_layers=3,hidden_size=128,activation='tanh',    epochs=10,learning_rate=0.005),
    make_args(optimizer='momentum', num_layers=3,hidden_size=64, activation='relu',    epochs=10,learning_rate=0.005,weight_decay=0.0005),
]
q7_rec = []

for i,args in enumerate(CFG_Q7):
    label = f'cfg{i+1}_{args.optimizer}_L{args.num_layers}_sz{args.hidden_size}_{args.activation}'
    print(f'\n▶  Q7 run {i+1}/12: {label}')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=f'q7_{label}', group='q7_global_performance',
                     config=vars(args), reinit=True)
    model,hist = train_model(args, Xtr_m, ytr_m, Xv_m, yv_m, run=run)
    tra = float(model.evaluate(Xtr_m, ytr_m))
    tea = float(model.evaluate(Xte_m, yte_m))
    run.log({'final_train_accuracy':tra,'test_accuracy':tea})
    run.finish()
    q7_rec.append({'label':label,'train_acc':tra,'test_acc':tea,
                   'optimizer':args.optimizer,'num_layers':args.num_layers,'activation':args.activation})

# Overlay + gap plots
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q7_overlay_plot', reinit=True)
tras=[r['train_acc'] for r in q7_rec]; teas=[r['test_acc'] for r in q7_rec]
gaps=[tr-te for tr,te in zip(tras,teas)]; x=list(range(len(q7_rec)))
xt=[f'run{i+1}' for i in x]
fig,axes=plt.subplots(1,2,figsize=(18,6))
axes[0].plot(x,tras,'o-',label='Train Acc',color='#377eb8',lw=2)
axes[0].plot(x,teas,'s--',label='Test Acc', color='#e41a1c',lw=2)
axes[0].fill_between(x,teas,tras,alpha=0.12,color='orange',label='Gap')
axes[0].set_xticks(x); axes[0].set_xticklabels(xt,rotation=45,ha='right')
axes[0].set_title('Train vs Test Accuracy — All 12 Runs'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True,alpha=0.3)
bcols=['#e41a1c' if g>0.05 else '#377eb8' for g in gaps]
axes[1].bar(xt,gaps,color=bcols)
axes[1].axhline(0.05,color='red',ls='--',lw=1.5,label='5% threshold')
axes[1].set_title('Train − Test Gap'); axes[1].legend()
axes[1].set_xticklabels(xt,rotation=45,ha='right'); axes[1].grid(True,alpha=0.3,axis='y')
plt.suptitle('Q2.7 — Overfitting Analysis',fontsize=13); plt.tight_layout()
run.log({'train_vs_test_overlay':wandb.Image(fig)}); plt.close(fig)
tbl=wandb.Table(columns=['run','optimizer','layers','activation','train_acc','test_acc','gap'])
for r,g in zip(q7_rec,gaps):
    tbl.add_data(r['label'],r['optimizer'],r['num_layers'],r['activation'],
                 round(r['train_acc'],4),round(r['test_acc'],4),round(g,4))
run.log({'performance_table':tbl})
run.finish()
print('Q2.7 done')


▶  Q7 run 1/12: cfg1_sgd_L2_sz64_relu


Built network: 2 hidden layers, 64 neurons, activation = relu
  Ep   1/10  loss=1.0334  tr=0.8709  val=0.8678
  Ep   2/10  loss=0.4001  tr=0.8997  val=0.8982
  Ep   3/10  loss=0.3276  tr=0.9114  val=0.9098
  Ep   4/10  loss=0.2911  tr=0.9194  val=0.9188
  Ep   5/10  loss=0.2661  tr=0.9274  val=0.9228
  Ep   6/10  loss=0.2464  tr=0.9327  val=0.9285
  Ep   7/10  loss=0.2302  tr=0.9372  val=0.9320
  Ep   8/10  loss=0.2164  tr=0.9403  val=0.9338
  Ep   9/10  loss=0.2046  tr=0.9420  val=0.9363
  Ep  10/10  loss=0.1942  tr=0.9456  val=0.9408


epoch,▁▂▃▃▄▅▆▆▇█
final_train_accuracy,▁
test_accuracy,▁
train_accuracy,▁▄▅▆▆▇▇███
train_loss,█▃▂▂▂▁▁▁▁▁
val_accuracy,▁▄▅▆▆▇▇▇██
val_loss,█▅▄▃▃▂▂▁▁▁
epoch,10
final_train_accuracy,0.94559
test_accuracy,0.9403
train_accuracy,0.94559



▶  Q7 run 2/12: cfg2_sgd_L4_sz128_sigmoid


Built network: 4 hidden layers, 128 neurons, activation = sigmoid
  Ep   1/10  loss=2.3053  tr=0.1124  val=0.1123
  Ep   2/10  loss=2.3022  tr=0.1124  val=0.1123
  Ep   3/10  loss=2.3019  tr=0.1044  val=0.1045
  Ep   4/10  loss=2.3017  tr=0.1124  val=0.1123
  Ep   5/10  loss=2.3011  tr=0.1891  val=0.1903
  Ep   6/10  loss=2.3007  tr=0.1022  val=0.1022
  Ep   7/10  loss=2.3007  tr=0.0991  val=0.0992
  Ep   8/10  loss=2.3001  tr=0.1124  val=0.1123
  Ep   9/10  loss=2.2997  tr=0.1124  val=0.1123
  Ep  10/10  loss=2.2990  tr=0.1124  val=0.1123


...


▶  Q7 run 12/12: cfg12_momentum_L3_sz64_relu


Built network: 3 hidden layers, 64 neurons, activation = relu
  Ep   1/10  loss=0.5303  tr=0.9238  val=0.9223
  Ep   2/10  loss=0.2189  tr=0.9473  val=0.9400
  Ep   3/10  loss=0.1670  tr=0.9576  val=0.9500
  Ep   4/10  loss=0.1363  tr=0.9621  val=0.9512
  Ep   5/10  loss=0.1174  tr=0.9701  val=0.9603
  Ep   6/10  loss=0.1023  tr=0.9751  val=0.9645
  Ep   7/10  loss=0.0908  tr=0.9785  val=0.9668
  Ep   8/10  loss=0.0821  tr=0.9794  val=0.9663
  Ep   9/10  loss=0.0741  tr=0.9832  val=0.9690
  Ep  10/10  loss=0.0694  tr=0.9836  val=0.9708


epoch,▁▂▃▃▄▅▆▆▇█
final_train_accuracy,▁
test_accuracy,▁
train_accuracy,▁▄▅▅▆▇▇███
train_loss,█▃▂▂▂▁▁▁▁▁
val_accuracy,▁▄▅▅▆▇▇▇██
val_loss,█▅▄▄▂▂▂▁▁▁
epoch,10
final_train_accuracy,0.98361
test_accuracy,0.9732
train_accuracy,0.98361


C:\Users\theri\AppData\Local\Temp\ipykernel_20104\3779612774.py:48: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[1].set_xticklabels(xt,rotation=45,ha='right'); axes[1].grid(True,alpha=0.3,axis='y')


Q2.7 done


---
## Q2.8 — Error Analysis

In [8]:
print('Training best model (20 epochs) ...')
best_args = make_args(optimizer='rmsprop', num_layers=3, hidden_size=128,
                      activation='relu', learning_rate=0.001,
                      weight_decay=0.0005, epochs=20)
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q8_best_model', reinit=True)
best_model, _ = train_model(best_args, Xtr_m, ytr_m, Xv_m, yv_m, run=run)
preds = best_model.predict(Xte_m); true = yte_m_raw
cm_m  = confusion_matrix(true, preds)

# 1. Confusion matrix
fig = plot_cm(cm_m, MNIST_CLASSES, title='Q2.8 — Confusion Matrix (Best Model · MNIST Test)')
run.log({'confusion_matrix': wandb.Image(fig)}); plt.close(fig)

# 2. Per-class accuracy
pca = cm_m.diagonal() / cm_m.sum(axis=1)
fig, ax = plt.subplots(figsize=(10,5))
cmap_r = plt.cm.RdYlGn
bars = ax.bar(MNIST_CLASSES, pca*100, color=[cmap_r(v) for v in pca])
ax.axhline(pca.mean()*100, color='navy', ls='--', lw=1.5, label=f'Mean {pca.mean()*100:.1f}%')
ax.set_title('Per-Class Accuracy'); ax.set_xlabel('Digit'); ax.set_ylabel('Acc %')
ax.set_ylim(0,107); ax.legend()
for bar,v in zip(bars,pca):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v*100:.1f}', ha='center',fontsize=9)
plt.tight_layout()
run.log({'per_class_accuracy': wandb.Image(fig)}); plt.close(fig)

# 3. Top-10 most confused pairs
confused = sorted([(cm_m[i,j],i,j) for i in range(10) for j in range(10) if i!=j],reverse=True)[:10]
fig, axes = plt.subplots(2,10,figsize=(20,5))
for col,(count,tc,pc) in enumerate(confused):
    mask=(true==tc)&(preds==pc); idxs=np.where(mask)[0]
    img = Xte_m[idxs[0]].reshape(28,28) if len(idxs) else np.zeros((28,28))
    axes[0][col].imshow(img,cmap='gray'); axes[0][col].axis('off')
    axes[0][col].set_title(f'T:{tc}\nP:{pc}',fontsize=8,color='red')
    axes[1][col].bar(['n'],[count],color='#e41a1c')
    axes[1][col].set_ylim(0,confused[0][0]*1.15)
    for sp in axes[1][col].spines.values(): sp.set_visible(False)
    axes[1][col].set_yticks([]); axes[1][col].set_xticks([])
    axes[1][col].set_title(f'n={count}',fontsize=8)
plt.suptitle('Q2.8 — Top 10 Most Confused Pairs',fontsize=13); plt.tight_layout()
run.log({'top10_confused_pairs': wandb.Image(fig)}); plt.close(fig)

run.log({'test_accuracy':float(accuracy_score(true,preds)),
         'test_f1':float(f1_score(true,preds,average='macro')),
         'test_precision':float(precision_score(true,preds,average='macro')),
         'test_recall':float(recall_score(true,preds,average='macro'))})
run.finish()
print('Q2.8 done')

Training best model (20 epochs) ...


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/20  loss=0.2671  tr=0.9626  val=0.9567
  Ep   2/20  loss=0.1192  tr=0.9732  val=0.9648
  Ep   3/20  loss=0.0907  tr=0.9706  val=0.9610
  Ep   4/20  loss=0.0779  tr=0.9815  val=0.9718
  Ep   5/20  loss=0.0696  tr=0.9846  val=0.9748
  Ep   6/20  loss=0.0629  tr=0.9837  val=0.9715
  Ep   7/20  loss=0.0608  tr=0.9764  val=0.9648
  Ep   8/20  loss=0.0567  tr=0.9864  val=0.9747
  Ep   9/20  loss=0.0548  tr=0.9731  val=0.9580
  Ep  10/20  loss=0.0525  tr=0.9839  val=0.9705
  Ep  11/20  loss=0.0504  tr=0.9900  val=0.9773
  Ep  12/20  loss=0.0493  tr=0.9862  val=0.9730
  Ep  13/20  loss=0.0479  tr=0.9876  val=0.9715
  Ep  14/20  loss=0.0454  tr=0.9888  val=0.9742
  Ep  15/20  loss=0.0467  tr=0.9901  val=0.9752
  Ep  16/20  loss=0.0436  tr=0.9819  val=0.9662
  Ep  17/20  loss=0.0440  tr=0.9867  val=0.9730
  Ep  18/20  loss=0.0433  tr=0.9887  val=0.9733
  Ep  19/20  loss=0.0426  tr=0.9917  val=0.9762
  Ep  20/20  loss=0.0412 

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_accuracy,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_accuracy,▁▄▃▆▆▆▄▇▄▆█▇▇▇█▆▇▇█▆
train_loss,█▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▄▂▆▇▆▄▇▁▆█▇▆▇▇▄▇▇█▅
val_loss,█▅▆▃▂▃▆▂█▃▂▂▃▂▁▅▂▂▁▃
epoch,20
test_accuracy,0.9698


Q2.8 done


---
## Q2.9 — Weight Initialization & Symmetry

In [9]:
NEURON_INDICES=[0,1,2,3,4]; N_ITER=50
INIT_CFGS=[{'weight_init':'zeros','label':'zeros_init'},{'weight_init':'xavier','label':'xavier_init'}]
q9_data={}

for cfg in INIT_CFGS:
    label=cfg['label']; print(f'\n▶  {label}')
    run=wandb.init(project=WANDB_PROJECT,entity=WANDB_ENTITY,
                   name=f'q9_{label}',group='q9_weight_init',config=cfg,reinit=True)
    args=make_args(weight_init=cfg['weight_init'],optimizer='rmsprop',activation='relu',
                   num_layers=3,hidden_size=128,learning_rate=0.001,epochs=5)
    model=NeuralNetwork(args); loss_fn,_=get_loss(args.loss)
    traces={n:[] for n in NEURON_INDICES}
    perm=np.random.permutation(Xtr_m.shape[0])
    Xs,ys=Xtr_m[perm],ytr_m[perm]
    for it,s in enumerate(range(0,Xtr_m.shape[0],args.batch_size)):
        if it>=N_ITER: break
        Xb=Xs[s:s+args.batch_size]; yb=ys[s:s+args.batch_size]
        fwd=model.forward(Xb); model.backward(yb,fwd); model.update_weights()
        g0=model.layers[0].grad_W; log_d={'iteration':it+1}
        for n_idx in NEURON_INDICES:
            gn=float(np.linalg.norm(g0[:,n_idx]))
            traces[n_idx].append(gn); log_d[f'neuron_{n_idx}_grad_norm']=gn
        run.log(log_d)
    q9_data[label]={n:list(v) for n,v in traces.items()}
    run.finish()

# Side-by-side plots
run=wandb.init(project=WANDB_PROJECT,entity=WANDB_ENTITY,
               name='q9_symmetry_comparison',reinit=True)
fig,axes=plt.subplots(1,2,figsize=(16,5))
ncols=plt.cm.tab10.colors
for ax,(label,trs) in zip(axes,q9_data.items()):
    for n_idx in NEURON_INDICES:
        ax.plot(range(1,len(trs[n_idx])+1),trs[n_idx],
                label=f'Neuron {n_idx}',color=ncols[n_idx],lw=2)
    ax.set_xlabel('Training Iteration'); ax.set_ylabel('||∇w||')
    ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
axes[0].set_title('Zeros Init — lines overlap (symmetry not broken)')
axes[1].set_title('Xavier Init — each neuron diverges uniquely')
plt.suptitle('Q2.9 — Symmetry Breaking: Zeros vs Xavier',fontsize=13); plt.tight_layout()
run.log({'symmetry_breaking_plot':wandb.Image(fig)}); plt.close(fig)
run.finish()
print('Q2.9 done')


▶  zeros_init


Built network: 3 hidden layers, 128 neurons, activation = relu


iteration,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
neuron_0_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_1_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_2_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_3_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_4_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
iteration,50
neuron_0_grad_norm,0
neuron_1_grad_norm,0
neuron_2_grad_norm,0
neuron_3_grad_norm,0



▶  xavier_init


Built network: 3 hidden layers, 128 neurons, activation = relu


iteration,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
neuron_0_grad_norm,▁▄▄▄▄███▄▅▄▂▃▄▅▃▆█▃▄▅▅▃▃▃▆▃▂▃▃▃▂▃▃▆▃▂▄▅▃
neuron_1_grad_norm,▁▁▂▁▁▆▃▅▃▇▁▃▂▄▄▇▂▆█▃▂▁▃▃▄▃▂█▃▃▃▄▆▄▃▂▃▂▃▃
neuron_2_grad_norm,▁▁▂▅▁▇▂▂▃▆▂▁▃▃▂▂▁▂█▂▂▂▂▃▂▃▇▄▃▂▄▃▂▂▅▂▃▃▃▅
neuron_3_grad_norm,▅▂▁▂▆▂▂▄▄▃▃▃▅▅▃▃▆▄▃▃▃▃▃▃▃█▂▇▆▆▆▂▂▃▇▄▇▃▃▃
neuron_4_grad_norm,▂▁▁▂▁▄▃▂▂▂▂▂▂▃▄▂▃▄▄▂▂▄▂▂▂▄▄▅█▃▅▂▂▃▃▃▃▃▃▄
iteration,50
neuron_0_grad_norm,0.14142
neuron_1_grad_norm,0.07493
neuron_2_grad_norm,0.19183
neuron_3_grad_norm,0.06191


Q2.9 done


## Q2.10 - Fashion-MNIST Transfer Challenge

In [8]:
# ── Q2.10 UPDATED — using actual top-3 configs from MNIST sweep ───────────────

CFGS_Q10 = [
    {
        'name':   'cfg1_momentum_relu_L3_sz128',
        'reason': 'Best MNIST sweep config (val_acc=0.9752): Momentum+ReLU+Xavier, 3 layers, 128 neurons, LR=0.01',
        'args':   make_args(
                      optimizer='momentum', activation='relu', weight_init='xavier',
                      num_layers=3, hidden_size=128, learning_rate=0.01,
                      weight_decay=0.0005, batch_size=128, epochs=20,
                      loss='cross_entropy'
                  )
    },
    {
        'name':   'cfg2_momentum_relu_L2_sz64',
        'reason': 'Second best MNIST sweep config (val_acc=0.9728): Momentum+ReLU+Xavier, 2 layers, 64 neurons, LR=0.01',
        'args':   make_args(
                      optimizer='momentum', activation='relu', weight_init='xavier',
                      num_layers=2, hidden_size=64, learning_rate=0.01,
                      weight_decay=0.0, batch_size=32, epochs=20,
                      loss='cross_entropy'
                  )
    },
    {
        'name':   'cfg3_rmsprop_relu_L2_sz64',
        'reason': 'Third best MNIST sweep config (val_acc=0.9725): RMSProp+ReLU+Xavier, 2 layers, 64 neurons, LR=0.005',
        'args':   make_args(
                      optimizer='rmsprop', activation='relu', weight_init='xavier',
                      num_layers=2, hidden_size=64, learning_rate=0.005,
                      weight_decay=0.0001, batch_size=128, epochs=20,
                      loss='cross_entropy'
                  )
    },
]

q10_res = []

for cfg in CFGS_Q10:
    name, args = cfg['name'], cfg['args']
    print(f'\n▶  {name}')
    print(f'   Reason: {cfg["reason"]}')

    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                     name=f'q10_{name}', group='q10_fashion_transfer',
                     config={**vars(args), 'reason': cfg['reason']},
                     reinit=True)

    model, hist = train_model(args, Xtr_f, ytr_f, Xv_f, yv_f, run=run)

    preds_f  = model.predict(Xte_f)
    ta = float(accuracy_score(yte_f_raw, preds_f))
    tf = float(f1_score(yte_f_raw, preds_f, average='macro'))
    run.log({'test_accuracy': ta, 'test_f1': tf})

    # confusion matrix for fashion classes
    cm_f = confusion_matrix(yte_f_raw, preds_f)
    fig  = plot_cm(cm_f, FASHION_CLASSES,
                   title=f'{name} — Fashion-MNIST Confusion Matrix',
                   cmap='Purples')
    run.log({'confusion_matrix_fashion': wandb.Image(fig)}); plt.close(fig)
    run.finish()

    q10_res.append({'name': name, 'test_acc': ta,
                    'test_f1': tf, 'reason': cfg['reason']})
    print(f'   Fashion Test Acc: {ta:.4f} | F1: {tf:.4f}')

# ── summary bar chart ─────────────────────────────────────────────────────────
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 name='q10_fashion_summary_v2', reinit=True)

accs = [r['test_acc'] for r in q10_res]
f1s  = [r['test_f1']  for r in q10_res]
ns   = [r['name']     for r in q10_res]
x    = np.arange(len(ns)); w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, accs, w, label='Test Accuracy',   color='#377eb8')
ax.bar(x + w/2, f1s,  w, label='Test F1 (macro)', color='#4daf4a')
ax.set_xticks(x)
ax.set_xticklabels([f'cfg{i+1}' for i in range(len(ns))], fontsize=10)
ax.set_title('Q2.10 — Fashion-MNIST Transfer: Top-3 Configs from MNIST Sweep',
             fontsize=12)
ax.set_ylabel('Score'); ax.legend(); ax.set_ylim(0, 1.08)
ax.grid(True, alpha=0.3, axis='y')
for xi, (a, f) in enumerate(zip(accs, f1s)):
    ax.text(xi - w/2, a + 0.005, f'{a:.3f}', ha='center', fontsize=9)
    ax.text(xi + w/2, f + 0.005, f'{f:.3f}', ha='center', fontsize=9)

# add config labels below x-axis
for xi, n in enumerate(ns):
    ax.text(xi, -0.07, n.replace('cfg', '').strip('_'),
            ha='center', fontsize=7, color='gray',
            transform=ax.get_xaxis_transform())
plt.tight_layout()
run.log({'fashion_transfer_summary': wandb.Image(fig)}); plt.close(fig)

# results table
tbl = wandb.Table(columns=['rank', 'config', 'mnist_val_acc',
                             'fashion_test_acc', 'fashion_f1', 'reasoning'])
mnist_val = [0.9752, 0.9728, 0.9725]
for i, (r, mv) in enumerate(zip(q10_res, mnist_val)):
    tbl.add_data(i+1, r['name'], mv,
                 round(r['test_acc'], 4),
                 round(r['test_f1'],  4),
                 r['reason'])
run.log({'fashion_results_table': tbl})
run.finish()
print('\n Q2.10 done')
print('\nSummary:')
for i, r in enumerate(q10_res):
    print(f'  cfg{i+1}: {r["name"]}')
    print(f'         Fashion Test Acc={r["test_acc"]:.4f}  F1={r["test_f1"]:.4f}')


▶  cfg1_momentum_relu_L3_sz128
   Reason: Best MNIST sweep config (val_acc=0.9752): Momentum+ReLU+Xavier, 3 layers, 128 neurons, LR=0.01


Built network: 3 hidden layers, 128 neurons, activation = relu
  Ep   1/20  loss=0.6761  tr=0.8255  val=0.8280
  Ep   2/20  loss=0.4348  tr=0.8579  val=0.8612
  Ep   3/20  loss=0.3931  tr=0.8694  val=0.8715
  Ep   4/20  loss=0.3665  tr=0.8792  val=0.8738
  Ep   5/20  loss=0.3515  tr=0.8773  val=0.8752
  Ep   6/20  loss=0.3376  tr=0.8756  val=0.8672
  Ep   7/20  loss=0.3215  tr=0.8925  val=0.8883
  Ep   8/20  loss=0.3097  tr=0.8932  val=0.8870
  Ep   9/20  loss=0.3027  tr=0.8954  val=0.8870
  Ep  10/20  loss=0.2951  tr=0.8960  val=0.8872
  Ep  11/20  loss=0.2883  tr=0.8873  val=0.8767
  Ep  12/20  loss=0.2808  tr=0.8954  val=0.8867
  Ep  13/20  loss=0.2720  tr=0.9029  val=0.8918
  Ep  14/20  loss=0.2694  tr=0.9117  val=0.8967
  Ep  15/20  loss=0.2628  tr=0.9099  val=0.8920
  Ep  16/20  loss=0.2597  tr=0.9151  val=0.8965
  Ep  17/20  loss=0.2516  tr=0.9109  val=0.8923
  Ep  18/20  loss=0.2470  tr=0.9067  val=0.8843
  Ep  19/20  loss=0.2445  tr=0.9159  val=0.8928
  Ep  20/20  loss=0.2353 

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_accuracy,▁
test_f1,▁
train_accuracy,▁▃▄▅▅▅▆▆▆▆▆▆▇█▇██▇██
train_loss,█▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val_accuracy,▁▄▅▆▆▅▇▇▇▇▆▇█████▇██
val_loss,█▅▄▃▃▄▂▂▂▂▃▂▂▁▁▁▂▂▁▁
epoch,20
test_accuracy,0.8834
test_f1,0.8828
train_accuracy,0.91735


   Fashion Test Acc: 0.8834 | F1: 0.8828

▶  cfg2_momentum_relu_L2_sz64
   Reason: Second best MNIST sweep config (val_acc=0.9728): Momentum+ReLU+Xavier, 2 layers, 64 neurons, LR=0.01


Built network: 2 hidden layers, 64 neurons, activation = relu
  Ep   1/20  loss=0.5577  tr=0.8492  val=0.8515
  Ep   2/20  loss=0.4049  tr=0.8634  val=0.8607
  Ep   3/20  loss=0.3673  tr=0.8680  val=0.8687
  Ep   4/20  loss=0.3430  tr=0.8879  val=0.8868
  Ep   5/20  loss=0.3267  tr=0.8915  val=0.8862
  Ep   6/20  loss=0.3128  tr=0.8909  val=0.8842
  Ep   7/20  loss=0.2998  tr=0.8933  val=0.8823
  Ep   8/20  loss=0.2904  tr=0.8996  val=0.8880
  Ep   9/20  loss=0.2819  tr=0.8981  val=0.8843
  Ep  10/20  loss=0.2761  tr=0.9037  val=0.8858
  Ep  11/20  loss=0.2650  tr=0.8999  val=0.8853
  Ep  12/20  loss=0.2602  tr=0.8985  val=0.8782
  Ep  13/20  loss=0.2577  tr=0.9047  val=0.8908
  Ep  14/20  loss=0.2485  tr=0.9115  val=0.8902
  Ep  15/20  loss=0.2471  tr=0.9098  val=0.8917
  Ep  16/20  loss=0.2386  tr=0.9148  val=0.8898
  Ep  17/20  loss=0.2350  tr=0.9169  val=0.8893
  Ep  18/20  loss=0.2319  tr=0.9156  val=0.8913
  Ep  19/20  loss=0.2272  tr=0.9212  val=0.8935
  Ep  20/20  loss=0.2231  

...

   Fashion Test Acc: 0.8797 | F1: 0.8793

▶  cfg3_rmsprop_relu_L2_sz64
   Reason: Third best MNIST sweep config (val_acc=0.9725): RMSProp+ReLU+Xavier, 2 layers, 64 neurons, LR=0.005


Built network: 2 hidden layers, 64 neurons, activation = relu
  Ep   1/20  loss=0.6162  tr=0.8276  val=0.8328
  Ep   2/20  loss=0.4309  tr=0.8630  val=0.8645
  Ep   3/20  loss=0.3974  tr=0.8720  val=0.8733
  Ep   4/20  loss=0.3767  tr=0.8578  val=0.8563
  Ep   5/20  loss=0.3661  tr=0.8617  val=0.8572
  Ep   6/20  loss=0.3578  tr=0.8703  val=0.8695
  Ep   7/20  loss=0.3494  tr=0.8711  val=0.8625
  Ep   8/20  loss=0.3452  tr=0.8847  val=0.8782
  Ep   9/20  loss=0.3405  tr=0.8872  val=0.8787
  Ep  10/20  loss=0.3362  tr=0.8914  val=0.8813
  Ep  11/20  loss=0.3347  tr=0.8851  val=0.8752
  Ep  12/20  loss=0.3340  tr=0.8799  val=0.8685
  Ep  13/20  loss=0.3346  tr=0.8942  val=0.8848
  Ep  14/20  loss=0.3278  tr=0.8822  val=0.8705
  Ep  15/20  loss=0.3269  tr=0.8728  val=0.8653
  Ep  16/20  loss=0.3272  tr=0.8846  val=0.8717
  Ep  17/20  loss=0.3262  tr=0.8722  val=0.8603
  Ep  18/20  loss=0.3247  tr=0.8806  val=0.8638
  Ep  19/20  loss=0.3251  tr=0.8857  val=0.8732
  Ep  20/20  loss=0.3205  

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_accuracy,▁
test_f1,▁
train_accuracy,▁▅▆▄▅▅▆▇▇█▇▆█▇▆▇▆▇▇▇
train_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▅▆▄▄▆▅▇▇█▇▆█▆▅▆▅▅▆▆
val_loss,█▃▃▆▄▃▅▂▁▁▃▃▁▃▄▂▆▃▃▃
epoch,20
test_accuracy,0.8605
test_f1,0.8579
train_accuracy,0.88481


   Fashion Test Acc: 0.8605 | F1: 0.8579



 Q2.10 done

Summary:
  cfg1: cfg1_momentum_relu_L3_sz128
         Fashion Test Acc=0.8834  F1=0.8828
  cfg2: cfg2_momentum_relu_L2_sz64
         Fashion Test Acc=0.8797  F1=0.8793
  cfg3: cfg3_rmsprop_relu_L2_sz64
         Fashion Test Acc=0.8605  F1=0.8579
